In [ ]:
# Rutas del proyecto (INPUT / OUTPUT)
function _resolver_base()
    for start in unique([(@__DIR__), pwd()])
        d = start
        for _ in 1:5
            isdir(joinpath(d, "INPUT", "24 NODE MODEL")) && return d
            nd = dirname(d); nd == d && break; d = nd
        end
    end
    return dirname(dirname(@__DIR__))
end
const BASE    = _resolver_base()
const IN_BESS = joinpath(BASE, "INPUT", "BESS")
const IN_WIND = joinpath(BASE, "INPUT", "Wind power generation forecast")
const IN_NODE = joinpath(BASE, "INPUT", "24 NODE MODEL")
const OUT_NB  = joinpath(BASE, "OUTPUT", "NOTEBOOKS")
for d in ["Basic","Eolico","BESS","Complete","UC"]; mkpath(joinpath(OUT_NB, d)); end
println("BASE del proyecto: ", BASE)


In [ ]:
#-------------------------LIBRERIAS----------------------------- No Contiene Eolico
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX


#-------------------------DATOS-----------------------------

# PARÁMETROS GENERALES DE GENERADORES
#Archivo de Generadores
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]

buses_gen      = Int64.(sheet["B"][2:end]) #Bus Generador
Pmax           = Float64.(sheet["C"][2:end]) #Potencia Maxima
Pmin           = Float64.(sheet["D"][2:end]) #Potencia Minima
Costo          = Float64.(sheet["E"][2:end]) #Costo por MW
Costofijo      = Float64.(sheet["F"][2:end]) #Costo Fijo 
CostoArranque  = Float64.(sheet["G"][2:end]) #Costo Arranque
nGen = length(Pmax) # Numero de Generadores
u0 = Int64.(sheet["H"][2:end]) # Estado inicial: 0 = apagado antes de t=1


# DEMANDA HORARIA
#Archivo de demanda
sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]

# Leer columna completa desde fila 2 en adelante 
demanda_horaria = Float64.(sheet_demanda["B"][2:end])

T = length(demanda_horaria) # Longitud del horizonte temporal


# RED Y LÍMITES DE LÍNEAS
#Archivo de Lineas de Transmision
xlsx_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))
sheet_lineas = xlsx_lineas[1]

# Detectar cuántas líneas hay 
nLineas = count(!ismissing, sheet_lineas["A"][2:end])

# Leer las columnas desde fila 2 hacia abajo
desde = Int64.(sheet_lineas["A"][2:1+nLineas])
hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
reactancia = Float64.(sheet_lineas["C"][2:1+nLineas])
flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])

# Crear diccionarios para reactancia y Fmax
x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
Sbase = 100.0 # Potencia base del sistema en MVA

nBuses = maximum(vcat(desde, hasta)) #Numero de bus del sistema

# PARÁMETROS DE BESS
# Leer el archivo y hoja
xlsx_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))
sheet_bess = xlsx_bess[1]

# Leer directamente los valores de la segunda fila
bus_bess         = Int(sheet_bess["A"][2])
Emax             = Float64(sheet_bess["B"][2])
pev_max          = Float64(sheet_bess["C"][2])
η_c              = Float64(sheet_bess["D"][2])
η_d              = Float64(sheet_bess["E"][2])
ramp_pev         = Float64(sheet_bess["F"][2])
SOC_ini          = Float64(sheet_bess["G"][2])
SOC_min_pct      = Float64(sheet_bess["H"][2])
pen_soc_low      = Float64(sheet_bess["I"][2])
carga_bess       = Float64(sheet_bess["J"][2])
descarga_bess    = Float64(sheet_bess["K"][2])
degradacion_bess = Float64(sheet_bess["L"][2])


# DISTRIBUCION DE CARGA 
xlsx_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))
sheet_carga = xlsx_carga[1]

# Leer columnas
buses_carga = Int64.(sheet_carga["A"][2:end])
porcentajes_carga = Float64.(sheet_carga["B"][2:end])

# Crear un diccionario para acceso rápido
distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))


#-------------------------MODELO-----------------------------
model = Model(HiGHS.Optimizer)


#-------------------------VARIABLES-----------------------------
# VARIABLES
#@variable(model, 0 <= p[g=1:nGen, t=1:T] <= Pmax[g])
@variable(model, p[g=1:nGen, t=1:T])
@variable(model, u[g=1:nGen, t=1:T], Bin)
@variable(model, v[g=1:nGen, t=1:T], Bin)
@variable(model, θ[b=1:nBuses, t=1:T])
@variable(model, 0 <= charge[t=1:T] <= pev_max)
@variable(model, 0 <= discharge[t=1:T] <= pev_max)
@variable(model, 0 <= soc[t=1:T] <= Emax)
@variable(model, pen_low[t=1:T] >= 0)   # penalización por SOC bajo
@variable(model, z[t=1:T], Bin)  # 1 si descarga, 0 si carga


#-------------------------RESTRICCIONES-----------------------------


#----------RESTRICCIONES DE BALANCE DE NODOS-------
# BALANCE NODAL
for t in 1:T
    for b in 1:nBuses
        gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
        gen += (b == bus_bess ? discharge[t] : 0.0)
        
        demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
        if b == bus_bess
            demanda_b += charge[t]
        end

        inflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
        outflow = sum(Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)

        @constraint(model, gen + inflow - outflow == demanda_b)
    end
end

#----------RESTRICCIONES DE GENERADORES-------
# RESTRICCIONES OPERATIVAS GENERADORES
for g in 1:nGen, t in 1:T
    @constraint(model, p[g,t] <= Pmax[g] * u[g,t])
    @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
end

# RESTRICCIONES DE ARRANQUE
for g in 1:nGen
    @constraint(model, v[g,1] >= u[g,1] - u0[g])
    @constraint(model, v[g,1] <= u[g,1])
    @constraint(model, v[g,1] <= 1 - u0[g])
    for t in 2:T
        @constraint(model, v[g,t] >= u[g,t] - u[g,t-1])
        @constraint(model, v[g,t] <= u[g,t])
        @constraint(model, v[g,t] <= 1 - u[g,t-1])
    end
end

#----------RESTRICCIONES DEL BESS-------
# SOC DINÁMICO Y RESTRICCIONES DEL BESS
@constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
for t in 2:T
    @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
    @constraint(model, charge[t] - charge[t-1] <= ramp_pev)
    @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
    @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev)
    @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
end

# Penalización por SOC bajo
for t in 1:T
    @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t])
end

#Evitar cargar y descargar el bess al mismo tiempo 
for t in 1:T
    @constraint(model, charge[t] <= pev_max * (1 - z[t]))
    @constraint(model, discharge[t] <= pev_max * z[t])
end

#----------RESTRICCIONES DE REDES-------
# LÍMITES DE FLUJO
for (i,j) in keys(x), t in 1:T
    fij = Sbase * (θ[i,t] - θ[j,t]) / x[(i,j)]
    @constraint(model, fij <= Fmax[(i,j)])
    @constraint(model, fij >= -Fmax[(i,j)])
end


# ÁNGULO DE REFERENCIA
for t in 1:T
    @constraint(model, θ[1,t] == 0)
end

# OBJETIVO
# CONDICIÓN TERMINAL DEL BESS: energía final ≥ inicial (operación sostenible día a día)
@constraint(model, soc[T] >= SOC_ini / 100 * Emax)

@objective(model, Min,
    sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T) +
    sum(pen_low[t]*pen_soc_low for t=1:T)
    + sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T)
    + sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T)
)

# OPTIMIZACIÓN
tic = time()
optimize!(model)
toc = time()

# RESULTADOS
status = termination_status(model)
println("\nEstado de optimización: ", status)
println("Tiempo de ejecución: ", round(toc - tic, digits=5), " s")

if status == MOI.OPTIMAL || status == MOI.FEASIBLE_POINT
    println("\nResumen horario de operación:")
    # Encabezado dinámico
    print("Hora\tDemanda")
    for g in 1:nGen
        print("\tG$g")
    end
    println("\tCharge\tDischarge\tSOC")
    
    # Filas de resultados
    for t in 1:T
        gvals = [round(value(p[g,t]), digits=2) for g in 1:nGen]
        print("$t\t", round(demanda_horaria[t], digits=2))
        for val in gvals
            print("\t", val)
        end
        println("\t", round(value(charge[t]), digits=2), "\t",
                round(value(discharge[t]), digits=2), "\t",
                round(value(soc[t]), digits=2))
    end
    println("\nCosto total: ", round(objective_value(model), digits=2))
else
    println("\nNo se encontró solución factible.")
end


In [ ]:
    println("\nResumen de demanda por nodo (MW):")
    for t in 1:T
        println("Hora $t:")
        for b in 1:nBuses
            demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
            if b == bus_bess
                demanda_b += value(charge[t])
            end
            println("  Nodo $b: ", round(demanda_b, digits=2))
        end
    end

    println("\nResumen de flujos en líneas (MW):")
    for t in 1:T
        println("Hora $t:")
        for (i,j) in keys(x)
            flujo = Sbase * (value(θ[i,t]) - value(θ[j,t])) / x[(i,j)]
            println("  Línea ($i → $j): ", round(flujo, digits=2))
        end
    end


In [ ]:
println("\nTabla de unidades activas por hora y capacidad máxima disponible:")

# Encabezado
header = ["Hora"]
append!(header, ["G$g" for g in 1:nGen])
append!(header, ["BESS", "Capacidad_Max", "Generación_Efectiva"])
println(join(header, "\t"))

for t in 1:T
    fila = [string(t)]
    capacidad_max = 0.0
    generacion_efectiva = 0.0

    for g in 1:nGen
        if round(value(u[g,t])) == 1
            push!(fila, "X")
            capacidad_max += Pmax[g]
            generacion_efectiva += value(p[g,t])
        else
            push!(fila, "")
        end
    end

    # Aportes eólica y BESS
    bess_val = round(value(discharge[t]), digits=2)

    capacidad_max +=  bess_val
    generacion_efectiva +=  bess_val

    push!(fila, string(bess_val)) # BESS
    push!(fila, string(round(capacidad_max, digits=2)))
    push!(fila, string(round(generacion_efectiva, digits=2)))

    println(join(fila, "\t"))
end


In [ ]:
using CSV, DataFrames

# Crear DataFrame con datos generales por hora
resultados_generales = DataFrame(
    Hora = 1:T,
    Demanda = demanda_horaria,
)

# Añadir generación de cada generador
for g in 1:nGen
    resultados_generales[!, "G$g"] = [value(p[g,t]) for t in 1:T]
end

# Añadir eólica, bess
resultados_generales[!, "Charge"] = [value(charge[t]) for t in 1:T]
resultados_generales[!, "Discharge"] = [value(discharge[t]) for t in 1:T]
resultados_generales[!, "SOC"] = [value(soc[t]) for t in 1:T]

# Guardar
CSV.write(joinpath(OUT_NB,"BESS","resultados_generales.csv"), resultados_generales)


# Crear DataFrame con demanda por nodo por hora
demanda_nodal = DataFrame()
demanda_nodal.Hora = 1:T
for b in 1:nBuses
    demanda_nodal[!, "Bus$b"] = [get(distribucion_carga, b, 0.0)*demanda_horaria[t] + (b == bus_bess ? value(charge[t]) : 0.0) for t in 1:T]
end
CSV.write(joinpath(OUT_NB,"BESS","demanda_nodal.csv"), demanda_nodal)


# Crear DataFrame con flujos por línea por hora
flujo_lineas = DataFrame(Hora = Int[])
for (i,j) in keys(x)
    flujo_lineas[!, "Linea_$(i)_$(j)"] = Float64[]
end

flujo_lineas = DataFrame(Hora = Float64[])
for (i,j) in keys(x)
    flujo_lineas[!, "Linea_$(i)_$(j)"] = Float64[]
end

for t in 1:T
    fila = [Float64(t)]
    for (i,j) in keys(x)
        flujo = Sbase * (value(θ[i,t]) - value(θ[j,t])) / x[(i,j)]
        push!(fila, flujo)
    end
    push!(flujo_lineas, permutedims(fila))  # añadir como fila
end

CSV.write(joinpath(OUT_NB,"BESS","flujos_lineas.csv"), flujo_lineas)


In [ ]:
using DataFrames, CSV

# Crear DataFrame con encabezados
header = ["Hora"]
append!(header, ["G$g" for g in 1:nGen])
append!(header, ["Capacidad_Max", "Generación_Efectiva"])

tabla_unidades = DataFrame()

for h in header
    tabla_unidades[!, h] = String[]  # Inicializar columnas como String
end

# Llenar la tabla con datos
for t in 1:T
    fila = [string(t)]
    capacidad_max = 0.0
    generacion_efectiva = 0.0

    for g in 1:nGen
        if round(value(u[g,t])) == 1
            push!(fila, "X")
            capacidad_max += Pmax[g]
            generacion_efectiva += value(p[g,t])
        else
            push!(fila, "")
        end
    end
    
    push!(fila, string(round(capacidad_max, digits=2)))
    push!(fila, string(round(generacion_efectiva, digits=2)))

    push!(tabla_unidades, fila)
end

# Guardar como CSV
CSV.write(joinpath(OUT_NB,"UC","UC_BESS.csv"), tabla_unidades)